# Past Presidents Dataset Generator

This project generates datasets of **past presidents of different countries** based on user input. It uses a quantized causal language model (transformers) and a Gradio interface for easy interaction.

**Features:**
- Select a country to generate a list of past presidents
- Optional: limit the number of entries or request a specific format (e.g., with terms/dates)
- 4-bit quantization via BitsAndBytes for lower memory usage
- Streaming generation with `TextStreamer`

### Imports

In [1]:
import gc
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig

### Model and quantization config

We use a small instruction-tuned model with **4-bit quantization** (BitsAndBytes) to reduce memory usage. For gated models (e.g. Phi), set `HF_TOKEN` in your environment or log in with `huggingface_hub.login()`.

In [2]:
MODEL_ID = "microsoft/Phi-4-mini-instruct"  # or "HuggingFaceH4/zephyr-7b-beta", "Qwen/Qwen2.5-1.5B-Instruct", etc.

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

### Prompt template and generation function

Build chat messages for the model and run generation with **TextStreamer** (streams tokens to the console). The full response is returned for the Gradio UI.

In [3]:
def build_presidents_prompt(country: str, max_entries: int = 12, include_dates: bool = True) -> list:
    """Build chat messages for generating a past presidents dataset for a given country."""
    detail = "Include each president's name, term start and end year (or dates), and a one-line fact." if include_dates else "List each president's name and a brief note."
    system_prompt = f"""You are a factual dataset generator for world history. When asked, you produce structured lists of past heads of state.

Rules:
1. Generate a list of past presidents (or equivalent heads of state) for the given country.
2. Include up to {max_entries} entries. Use historically accurate names and dates.
3. {detail}
4. Use a clear numbered list. Keep each entry to one short line. Be concise.
5. Only include past presidents (no current acting leaders unless you clearly label them).
6. Do not write long paragraphs; output only the list."""

    user_content = f"Generate a dataset of past presidents of {country}. Maximum {max_entries} entries."
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]

In [4]:
def generate_presidents_dataset(country: str, max_entries: int = 10, include_dates: bool = True) -> str:
    """Generate a past presidents dataset using a quantized causal LM and TextStreamer."""
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        messages = build_presidents_prompt(country, max_entries=max_entries, include_dates=include_dates)

        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        tokenizer.pad_token = tokenizer.eos_token

        input_ids = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            truncation=True,
            max_length=2048,
        ).to(device)

        attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)

        if device == "cuda" and quant_config is not None:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID,
                quantization_config=quant_config,
                device_map="auto",
            )
        elif device == "cuda":
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
        else:
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)

        streamer = TextStreamer(tokenizer, skip_special_tokens=True, skip_prompt=True)

        with torch.inference_mode():
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=1024,
                do_sample=True,
                temperature=0.6,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
                streamer=streamer,
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "<|assistant|>" in generated_text:
            result = generated_text.split("<|assistant|>")[-1].strip()
        elif "assistant" in generated_text.lower():
            parts = generated_text.split("assistant")[-1].strip()
            result = parts if parts else generated_text
        else:
            result = generated_text

        del model
        torch.cuda.empty_cache()
        gc.collect()

        return result.strip()

    except Exception as e:
        return f"Error generating dataset: {str(e)}"

### Gradio interface

Launch the app to generate past presidents datasets by country. Uses the quantized model and streams tokens to the console via `TextStreamer` while the UI shows the final result.

In [5]:
def create_presidents_interface():
    with gr.Blocks(title="Past Presidents Dataset Generator", theme=gr.themes.Soft()) as demo:
        gr.Markdown("# Past Presidents Dataset Generator")
        gr.Markdown("Generate a dataset of past presidents (or heads of state) for any country. Uses a **quantized** causal LM and **TextStreamer**.")

        with gr.Row():
            with gr.Column():
                country_input = gr.Textbox(
                    label="Country",
                    placeholder="e.g. United States, Nigeria, France",
                    value="United States",
                    info="Country for which to list past presidents",
                )
                max_entries_slider = gr.Slider(
                    minimum=3,
                    maximum=15,
                    step=1,
                    value=8,
                    label="Max entries",
                    info="Maximum number of presidents to list",
                )
                include_dates_check = gr.Checkbox(
                    label="Include term dates and short fact",
                    value=True,
                )
                generate_btn = gr.Button("Generate dataset", variant="primary", size="lg")

            with gr.Column():
                output_text = gr.Textbox(
                    label="Generated dataset",
                    lines=20,
                    placeholder="Generated list will appear here...",
                    show_copy_button=True,
                )

        gr.Markdown("""
        ### How it works
        - **Model**: Loaded with 4-bit quantization (`BitsAndBytesConfig`) to reduce VRAM.
        - **Generation**: Uses `AutoModelForCausalLM.generate()` with `TextStreamer` so tokens stream in the console.
        - Results are based on the model's knowledge; verify important dates and names from authoritative sources.
        """)

        generate_btn.click(
            fn=generate_presidents_dataset,
            inputs=[country_input, max_entries_slider, include_dates_check],
            outputs=output_text,
        )

        return demo

In [6]:
demo = create_presidents_interface()
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
